# 本地 LLM Vision（OpenAI 兼容，例如 LM Studio）

- 服务根地址：`http://127.0.0.1:1234`；Python 里请把 **base_url** 设为 `http://127.0.0.1:1234/v1`（多出来的 `/v1` 是 OpenAI 兼容路由）。
- 在 LM Studio 中需**加载支持视觉的多模态模型**，并把下方代码里的 **`MODEL`** 改为对应的模型 id。
- 示例从本地文件读入图片，再编码为 base64 放入 `data:` URL（不要把文件路径当作 base64 拼进 URL）。

In [ ]:
%pip install -q openai

In [ ]:
from openai import OpenAI

# 服务根地址一般是 http://127.0.0.1:1234 ，OpenAI 兼容接口需在路径上加 /v1
BASE_URL = "http://127.0.0.1:1234/v1"
API_KEY = "lm-studio"
MODEL = "qwen/qwen3-vl-4b"

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)


In [ ]:
# 读本地图片：读入字节后做 base64，再放进 data: URL（不要把「文件路径」当成 base64 字符串）
import base64
from pathlib import Path

IMAGE_NAME = "微信图片_20260504122723_232_510.jpg"
USER_PROMPT = "请用一句话描述这张图片（若看不清则说明即可）。"


def resolve_local_image(name: str) -> Path:
    """兼容 cwd 为仓库根目录或 notebooks/ 等情况。"""
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        for rel in (
            base / "notebooks" / "19_image" / name,
            base / "19_image" / name,
        ):
            if rel.is_file():
                return rel.resolve()
    raise FileNotFoundError(
        f"找不到 {name}。当前 cwd: {here}。请确认文件在 notebooks/19_image/ 下，或把 IMAGE_PATH 改成绝对路径。"
    )


def print_message(
    tag: str,
    *,
    text: str | None = None,
    image_path: str | None = None,
    message=None,
    resp=None,
) -> None:
    print(f"\n--- [{tag}] ---")
    if text is not None:
        print(text)
    if image_path is not None:
        print(f"image: {image_path}")
    if message is not None:
        print("content:\n", message.content)
    if resp is not None:
        ch0 = resp.choices[0]
        print(
            "response_metadata:",
            f"model={resp.model}",
            f"finish_reason={ch0.finish_reason}",
            f"usage={resp.usage}",
        )


IMAGE_PATH = resolve_local_image(IMAGE_NAME)
suffix = IMAGE_PATH.suffix.lower()
mime = (
    "image/jpeg"
    if suffix in {".jpg", ".jpeg"}
    else "image/png"
    if suffix == ".png"
    else "application/octet-stream"
)

b64 = base64.standard_b64encode(IMAGE_PATH.read_bytes()).decode("ascii")
image_url = f"data:{mime};base64,{b64}"

print_message("用户", text=USER_PROMPT, image_path=str(IMAGE_PATH))

resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": USER_PROMPT},
                {"type": "image_url", "image_url": {"url": image_url}},
            ],
        }
    ],
    max_tokens=256,
)

print_message("模型", message=resp.choices[0].message, resp=resp)
print("\n=== 结束 ===")
